## **Python Code**

In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold,cross_val_score

In [2]:
iris = load_iris() # load data into memory
x,y = iris.data ,iris.target # features and target arrays
x[:5]

array([[5.1, 3.5, 1.4, 0.2],
       [4.9, 3. , 1.4, 0.2],
       [4.7, 3.2, 1.3, 0.2],
       [4.6, 3.1, 1.5, 0.2],
       [5. , 3.6, 1.4, 0.2]])

In [3]:
y[:5]

array([0, 0, 0, 0, 0])

In [4]:
model = LogisticRegression(max_iter=200,n_jobs=None)
# maximum number of iterations the solver will run to converge
# sometimes Logistic regression needs more iterations for convergence(when a model and algorithm gradually reaches a  stable and correct solution after repeated steps)

# number of cpu core to use for parallel processing.
# none = use 1 core (default)
# -1 = use all avilable core for faster computation
cv = KFold(n_splits=5,shuffle=True,random_state=42)
# random state : seed for reoroducibility

scores = cross_val_score(model,x,y,cv = cv ,scoring='accuracy') # run CV and collect accuracy
# evaluation matric : here we use accuracy

In [5]:
print("Fold accuracies: ",scores) # show each fold's score
print('Mean accuracy: ',scores.mean()) # average performence (central tendency)
print('Std dev: ',scores.std()) # varibability across folds
print('Approx 95% CI: [{:.3f},{:.3f}]'.format(scores.mean()-1.96*scores.std()/np.sqrt(len(scores)),scores.mean()+1.96*scores.std()/np.sqrt(len(scores))))

Fold accuracies:  [1.         1.         0.93333333 0.96666667 0.96666667]
Mean accuracy:  0.9733333333333334
Std dev:  0.024944382578492935
Approx 95% CI: [0.951,0.995]


## **Note:**
We'll use the breast cancer dataset and evaluate a logistic regression model . instad of looking at just accuracy, we'll measure multiple metrices - precision , recall ,F1,and ROC-AUC - using 10 dolds stratified cross-validation . this gives us a fair and well - rounded view of how reliable the model is

### **Proper evaluation with stratified KFold + multiple matrices + Pipeline(to avoid leakage)<br> We'll scale features inside a pipeline , evaluate multiple metrices and compute out-of-fold predictions**

In [6]:
import numpy as np
from sklearn.datasets import load_breast_cancer # binary classification dataset
from sklearn.pipeline import Pipeline # to chain preprocessing + model saftely
from sklearn.preprocessing import StandardScaler # feature scaling
from sklearn.linear_model import LogisticRegression # linear classifier 
from sklearn.metrics import confusion_matrix,classification_report,roc_auc_score # evaluation metrices
from sklearn.model_selection import StratifiedKFold,cross_validate

In [7]:
data = load_breast_cancer()
x, y = data.data,data.target
x[:5]

array([[1.799e+01, 1.038e+01, 1.228e+02, 1.001e+03, 1.184e-01, 2.776e-01,
        3.001e-01, 1.471e-01, 2.419e-01, 7.871e-02, 1.095e+00, 9.053e-01,
        8.589e+00, 1.534e+02, 6.399e-03, 4.904e-02, 5.373e-02, 1.587e-02,
        3.003e-02, 6.193e-03, 2.538e+01, 1.733e+01, 1.846e+02, 2.019e+03,
        1.622e-01, 6.656e-01, 7.119e-01, 2.654e-01, 4.601e-01, 1.189e-01],
       [2.057e+01, 1.777e+01, 1.329e+02, 1.326e+03, 8.474e-02, 7.864e-02,
        8.690e-02, 7.017e-02, 1.812e-01, 5.667e-02, 5.435e-01, 7.339e-01,
        3.398e+00, 7.408e+01, 5.225e-03, 1.308e-02, 1.860e-02, 1.340e-02,
        1.389e-02, 3.532e-03, 2.499e+01, 2.341e+01, 1.588e+02, 1.956e+03,
        1.238e-01, 1.866e-01, 2.416e-01, 1.860e-01, 2.750e-01, 8.902e-02],
       [1.969e+01, 2.125e+01, 1.300e+02, 1.203e+03, 1.096e-01, 1.599e-01,
        1.974e-01, 1.279e-01, 2.069e-01, 5.999e-02, 7.456e-01, 7.869e-01,
        4.585e+00, 9.403e+01, 6.150e-03, 4.006e-02, 3.832e-02, 2.058e-02,
        2.250e-02, 4.571e-03, 2.357e

In [8]:
y[:5]

array([0, 0, 0, 0, 0])

In [9]:
# difine pipeline 
pipe = Pipeline([
    ('scaler',StandardScaler(),
     ('clf',LogisticRegression(max_iter=500)))
])

# purpose:
# - prevents dataleakage(scaler is fit only on training data in each CV split)
# - keeps workflow clean (all steps chained togather)
# - Makes model evaluation & hyperparemeter tuning easier

In [10]:
# 3) Difine CV strategy
cv = StratifiedKFold(n_splits=10,shuffle=True,random_state=42) #preserves class balance

In [11]:
# 4) Define scoring matrices
scoring = {
    'accuracy': 'accuracy',
    "Precision":'Precision',
    'recall': 'recall',
    'f1':'f1',
    'roc_auc': 'roc_auc'

}

In [12]:
# Run cross validation 
cv_results = cross_validate(
    pipe,x,y,cv = cv,scoring=scoring,return_train_score=False
)

ValueError: 'Precision' is not a valid scoring value. Use sklearn.metrics.get_scorer_names() to get valid options.

In [13]:
# 6) display results 
for metric in scoring.keys():
    values = cv_results[f'test_{metric}']
    print(f'{metric:>9}: {values.mean():.3f}+- {values.std():.3f}')

NameError: name 'cv_results' is not defined

In [14]:
from sklearn.model_selection import GridSearchCV

In [15]:
iris = load_iris()
x,y = iris.data,iris.target

In [16]:
pipe = Pipeline([
    ('scalar',StandardScaler()),
    ('clf',LogisticRegression(max_iter=500))
])

In [24]:
# Difine hyperparemeter grid
paren_grid = {
    'clf_penalty':['l1','l2'], # Type of regularization 
    'clf_C':[0.01, 0.1,1,10] ,#regularization strength
    'clf_solver':['liblinear'] # solver that supports L1/L2
}
      

In [21]:
grid = GridSearchCV(
    estimator=pipe,
    param_grid=paren_grid,
    cv=5,
    scoring='accuracy',
    refit=True,
    n_jobs=-1
)

In [25]:
# run the grid search
grid.fit(x,y)
print('Best Paremeters:',grid.best_params_)
print('Best Cross-Validation Accuracy:', grid.best_score_)
print('Test Accuracy with Best Model:',grid.score(x,y))

ValueError: Invalid parameter 'clf_C' for estimator Pipeline(steps=[('scalar', StandardScaler()),
                ('clf', LogisticRegression(max_iter=500))]). Valid parameters are: ['memory', 'steps', 'transform_input', 'verbose'].